# Run the full pipeline — no command line needed

This notebook does everything the CLI scripts (`src/train.py`, `src/evaluate.py`)
do, but cell-by-cell so you can see and run each step yourself: clean the data,
build the preprocessing pipeline, train both models, save them, and evaluate them.

It still calls the real functions in `src/` (not copies of the logic) — so what
runs here is exactly what runs in the CLI scripts and in production, just shown
one step at a time instead of hidden inside one script.

**You do not need to run `python src/train.py` before this notebook — this notebook
replaces that step.** Once you've run all the cells here, `eda_and_walkthrough.ipynb`
will also work, since the models will be saved to `models/`.

In [1]:
%matplotlib inline
import sys
sys.path.insert(0, '../src')

import joblib
from pathlib import Path

from data import load_and_clean
from features import build_preprocess_pipeline
from train import (
    build_classification_split, build_regression_split,
    train_classifier, train_regressor,
    DEFAULT_CLF_PARAMS, DEFAULT_REG_PARAMS,
)
from evaluate import evaluate_classifier, evaluate_regressor, shap_feature_importance

MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

z:\aplus_workshop\insurance-portfolio\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1 — Load and clean the raw data

This is the cleaning step: renames the original dataset's cryptic column names
(e.g. `KIDSDRIV` -> `num_young_drivers`), strips `$`/`,` out of currency columns,
removes duplicate rows, and drops the ID/date-of-birth columns. See `src/data.py`
if you want to see exactly what it does.

In [2]:
df = load_and_clean('../data/car_insurance_claim.csv')
print('Cleaned shape:', df.shape)
df.head()

Cleaned shape: (10301, 25)


,num_young_drivers,age,num_of_children,years_job_held_for,income,single_parent,value_of_home,married,gender,highest_education,...,vehicle_type,red_vehicle,5_year_total_claims_value,5_year_num_of_claims,licence_revoked,license_points,new_claim_value,vehicle_age,is_claim,address_type
0,0,60.0,0,11.0,67349,No,0,No,M,PhD,...,Minivan,yes,4461,2,No,3,0,18.0,0,Highly Urban/ Urban
1,0,43.0,0,11.0,91449,No,257252,No,M,High School,...,Minivan,yes,0,0,No,0,0,1.0,0,Highly Urban/ Urban
2,0,48.0,0,11.0,52881,No,0,No,M,Bachelors,...,Van,yes,0,0,No,2,0,10.0,0,Highly Urban/ Urban
3,0,35.0,1,10.0,16039,No,124191,Yes,F,High School,...,SUV,no,38690,2,No,3,0,10.0,0,Highly Urban/ Urban
4,0,51.0,0,14.0,<NA>,No,306251,Yes,M,<High School,...,Minivan,yes,0,0,No,0,0,6.0,0,Highly Urban/ Urban


## Step 2 — Split into train/test

Two separate splits: one for the classifier (does a claim occur — uses the whole
dataset), one for the regressor (how large is the claim — only rows where a claim
actually occurred). Both are stratified/split the same way `src/train.py` does it,
so results here match what the CLI would produce.

In [3]:
X_train, X_test, y_train, y_test = build_classification_split(df)
Xr_train, Xr_test, yr_train, yr_test = build_regression_split(df)

print('Classification train/test:', X_train.shape, X_test.shape)
print('Regression train/test:', Xr_train.shape, Xr_test.shape)

Classification train/test: (8240, 23) (2061, 23)
Regression train/test: (2196, 23) (550, 23)


## Step 3 — The feature engineering / preprocessing pipeline

This is the step that turns raw columns into model-ready numbers: KNN imputation
for missing values, a square-root transform on skewed numeric columns, ordinal
encoding for education, and one-hot encoding for occupation/vehicle type. Let's
look at what it actually produces before training anything, just to see it work.

In [4]:
preview_pipeline = build_preprocess_pipeline()
X_train_prepared_preview = preview_pipeline.fit_transform(X_train)
print('Shape after preprocessing:', X_train_prepared_preview.shape)
X_train_prepared_preview.head()

Shape after preprocessing: (8240, 32)


,num__num_young_drivers,num__age,num__num_of_children,num__years_job_held_for,num__income,num__value_of_home,num__commute_dist,num__vehicle_value,num__policy_tenure,num__5_year_total_claims_value,...,cat_one_hot__occupation_Home Maker,cat_one_hot__occupation_Lawyer,cat_one_hot__occupation_Manager,cat_one_hot__occupation_Professional,cat_one_hot__occupation_Student,cat_one_hot__vehicle_type_Panel Truck,cat_one_hot__vehicle_type_Pickup,cat_one_hot__vehicle_type_SUV,cat_one_hot__vehicle_type_Sports Car,cat_one_hot__vehicle_type_Van
8452,-0.332668,1.409817,-0.643344,-2.563674,-0.375723,-1.452458,0.498062,-1.040804,1.139483,-0.461230,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
7822,3.669060,-0.453096,2.054902,0.874127,-0.866152,0.037836,-0.479232,-1.159364,-0.142286,-0.461230,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1312,-0.332668,-1.850281,1.155487,0.383013,0.090791,0.424047,0.443522,-2.392885,-0.142286,0.161972,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
921,-0.332668,0.129064,0.256071,0.628570,0.258015,0.569484,-0.929180,0.845452,-1.245093,0.147137,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
8708,-0.332668,-0.685960,-0.643344,-0.599216,1.093996,1.062875,-0.409978,-1.374057,-1.245093,-0.461230,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0


## Step 4 — Train the classifier (is a claim filed?)

`train_classifier()` builds one sklearn `Pipeline` combining the preprocessing
step above with an `XGBClassifier`, and fits the whole thing on the training data
only. This is the structural fix for the original notebook's test-leakage bug —
the preprocessing step only ever sees training data during `.fit()`.

In [5]:
clf_pipeline = train_classifier(X_train, y_train, params=DEFAULT_CLF_PARAMS)
print('Classifier trained.')
clf_pipeline

Classifier trained.


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](23,)","['num_young_drivers','age','num_of_children',...,'license_points', 'vehicle_age','address_type']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,23
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('drop_features', ...), ('num', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specif

## Step 5 — Train the regressor (how large is the claim?)

In [6]:
reg_pipeline = train_regressor(Xr_train, yr_train, params=DEFAULT_REG_PARAMS)
print('Regressor trained.')
reg_pipeline

Regressor trained.


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](23,)","['num_young_drivers','age','num_of_children',...,'license_points', 'vehicle_age','address_type']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,23
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('drop_features', ...), ('num', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically 

## Step 6 — Save the trained models

This writes to `models/`, exactly like `python src/train.py` does — after this
cell, the models exist on disk and `eda_and_walkthrough.ipynb` (or the FastAPI
service in `src/serve.py`) can load them.

In [7]:
joblib.dump(clf_pipeline, MODELS_DIR / 'classifier_pipeline.joblib')
joblib.dump(reg_pipeline, MODELS_DIR / 'regressor_pipeline.joblib')
print('Saved both pipelines to', MODELS_DIR.resolve())

Saved both pipelines to Z:\aplus_workshop\insurance-portfolio\models


## Step 7 — Evaluate on the held-out test set

Calling `.predict()`/`.predict_proba()` on the fitted pipelines only ever
`.transform()`s the test data (never re-fits on it) — this is what makes these
numbers a true, leakage-free estimate.

In [8]:
clf_metrics = evaluate_classifier(clf_pipeline, X_test, y_test)
clf_metrics

{'f1_weighted': 0.7815087738694976,
 'roc_auc': 0.8217558047537153,
 'pr_auc_average_precision': 0.642804222678808,
 'positive_class_rate_test': 0.27365356622998543}

In [9]:
reg_metrics = evaluate_regressor(reg_pipeline, Xr_test, yr_test)
reg_metrics

{'rmse': 8381.073880450047,
 'mae': 3549.3347349297424,
 'baseline_rmse': 8285.106663145973,
 'baseline_mae': 3309.639233057851,
 'rmse_improvement_pct': -1.158309979652505}

**Note on the regressor result:** compare `rmse` against `baseline_rmse` above —
they're close, and the tuned model doesn't clearly beat a naive mean-prediction
baseline. That's a real, honestly-reported finding about this dataset, not a bug —
see `DECISIONS.md` for the full explanation.

## Step 8 — Feature importance (SHAP)

What's actually driving each model's predictions.

In [10]:
clf_importance = shap_feature_importance(
    clf_pipeline, X_test.sample(min(300, len(X_test)), random_state=42), top_n=10,
)
import pandas as pd
pd.DataFrame(clf_importance)

,feature,mean_abs_shap
0,cat_bin__address_type,0.491947
1,cat_bin__type_of_use,0.344855
2,num__5_year_total_claims_value,0.235773
3,cat_ord__highest_education,0.210576
4,num__age,0.209121
5,cat_bin__married,0.206495
6,num__value_of_home,0.203952
7,num__policy_tenure,0.199758
8,num__commute_dist,0.193731
9,cat_bin__licence_revoked,0.191989


In [11]:
reg_importance = shap_feature_importance(
    reg_pipeline, Xr_test.sample(min(300, len(Xr_test)), random_state=42), top_n=10,
)
pd.DataFrame(reg_importance)

Background dataset has 300 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=300 when initializing the masker.


,feature,mean_abs_shap
0,num__vehicle_value,729.978207
1,cat_bin__gender,585.562991
2,num__vehicle_age,544.358171
3,cat_ord__highest_education,419.448188
4,cat_one_hot__vehicle_type_SUV,381.993762
5,cat_bin__single_parent,297.033765
6,num__policy_tenure,219.839331
7,cat_bin__licence_revoked,187.196099
8,cat_one_hot__occupation_Professional,153.202765
9,num__age,149.309567


## Optional — hyperparameter tuning with Optuna

This is slower (does 5-fold cross-validation per trial) and tracks every trial in
MLflow. Skip this cell if you just want the default-hyperparameter models above.
Uncomment to run it.

In [13]:
from tune import tune_classifier, tune_regressor
import mlflow
mlflow.set_tracking_uri('sqlite:///../mlflow.db')
mlflow.set_experiment('insurance_claim_portfolio')

best_clf_params, best_clf_score = tune_classifier(X_train, y_train, n_trials=15)
print('Best classifier CV PR-AUC:', best_clf_score)
print('Best params:', best_clf_params)

# Retrain with the tuned params and re-save, same as Steps 4/6 above:
clf_pipeline = train_classifier(X_train, y_train, params=best_clf_params)
joblib.dump(clf_pipeline, MODELS_DIR / 'classifier_pipeline.joblib')

2026/09/16 23:44:24 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/16 23:44:24 INFO mlflow.store.db.utils: Updating database tables
2026/09/16 23:44:28 INFO mlflow.tracking.fluent: Experiment with name 'insurance_claim_portfolio' does not exist. Creating a new experiment.
[I 2026-09-16 23:44:29,007] A new study created in memory with name: classifier_tuning
[I 2026-09-16 23:44:37,910] Trial 0 finished with value: 0.6325124421081567 and parameters: {'n_estimators': 128, 'max_depth': 3, 'learning_rate': 0.19562843331277757, 'subsample': 0.6700329791493987, 'colsample_bytree': 0.9301908523905412, 'min_child_weight': 6, 'gamma': 0.9947936314105741, 'reg_alpha': 0.0004153024758861694, 'reg_lambda': 8.229203466553392}. Best is trial 0 with value: 0.6325124421081567.
[I 2026-09-16 23:44:44,838] Trial 1 finished with value: 0.5864439010586782 and parameters: {'n_estimators': 117, 'max_depth': 5, 'learning_rate': 0.2822996780990836, 'subsample': 0.654547842038534

Best classifier CV PR-AUC: 0.6376101434251593
Best params: {'n_estimators': 313, 'max_depth': 6, 'learning_rate': 0.021692753688682165, 'subsample': 0.9272982501603086, 'colsample_bytree': 0.8478691453298082, 'min_child_weight': 13, 'gamma': 0.011998184497560183, 'reg_alpha': 0.009128968368556414, 'reg_lambda': 0.0010720944556892112}


['..\\models\\classifier_pipeline.joblib']

## Done

The models are trained and saved. From here you can:
- Open `eda_and_walkthrough.ipynb` for more EDA and a second look at these same results
- Run `uvicorn src.serve:app --reload --app-dir ../src` (from `notebooks/`, adjust
  the path) to serve these models over an API
- Re-run this notebook any time you want to retrain from scratch